In [ ]:
import math
import importlib
import load_data_from_cache
import CNN_src as cnn
import load_data_from_cache as cache_loader
import Tansformer_src as ts
from torch.utils.data import DataLoader
import torch

train_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/train")
train_emb_loader = DataLoader(train_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
val_emb_ds = cache_loader.CachedEmbeddingDataset('cache/esm2_t30/val')
val_emb_loader = DataLoader(val_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
test_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/test")
test_emb_loader = DataLoader(test_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
    
model = cnn.CNNHead(
    d_in=640,
    hidden=64,
    mlp_hidden=32,
    n_layers=1,
    kernel_size=3,
    dropout=0.05,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)

for epoch in range(1, 21):
    tr = cnn.train_one_epoch(model, train_emb_loader, optimizer, device)
    va_loss, va_r = cnn.eval_one_epoch(model, val_emb_loader, device, clamp_for_metrics=True)
    print(f"epoch {epoch:02d} | train {tr:.5f} | val {va_loss:.5f} | val pearson {va_r:.4f}")